# BiciTierra Market · Explorar dades abans de predir

Dades sintètiques: 66 mesos i 18 sèries comercials. Objectiu: predir unitats d’un mes abans que comence. Llegiu el [diccionari](../dades/DICCIONARI_DADES.md).

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path('..') / 'data'
sales = pd.read_csv(DATA / 'sales_history.csv')
sales['month'] = pd.to_datetime(sales['month'], format='%Y-%m')
KEY = ['channel', 'product_line', 'region']
sales = sales.sort_values(KEY + ['month']).reset_index(drop=True)
assert not sales.duplicated(['month'] + KEY).any()
sales.head()
customers = pd.read_csv(DATA / 'customers.csv')
print('Files de vendes:', len(sales), 'Clients:', len(customers))

Files de vendes: 1188 Clients: 2000


## 1. Estructura, cobertura i qualitat

Cada fila de vendes és un mes, canal, producte i regió. Comproveu la cobertura abans de crear retards.

In [3]:
coverage = sales.groupby(KEY)['month'].agg(['min', 'max', 'nunique'])
print(coverage)
print(sales.isna().sum())
print(customers.isna().sum())
print('Duplicats de clau:', sales.duplicated(['month'] + KEY).sum())

                                     min        max  nunique
channel   product_line region                               
ecommerce electric     coast  2021-01-01 2026-06-01       66
                       inland 2021-01-01 2026-06-01       66
                       metro  2021-01-01 2026-06-01       66
          mountain     coast  2021-01-01 2026-06-01       66
                       inland 2021-01-01 2026-06-01       66
                       metro  2021-01-01 2026-06-01       66
          urban        coast  2021-01-01 2026-06-01       66
                       inland 2021-01-01 2026-06-01       66
                       metro  2021-01-01 2026-06-01       66
store     electric     coast  2021-01-01 2026-06-01       66
                       inland 2021-01-01 2026-06-01       66
                       metro  2021-01-01 2026-06-01       66
          mountain     coast  2021-01-01 2026-06-01       66
                       inland 2021-01-01 2026-06-01       66
                       m

## 2. Separar el temps abans de prendre decisions de model

Exploreu patrons amb entrenament. Reserveu 2025 per comparar decisions i 2026 per a la prova final. No és una partició aleatòria.

In [4]:
train = sales[sales['month'] < '2025-01-01'].copy()
validation = sales[(sales['month'] >= '2025-01-01') & (sales['month'] < '2026-01-01')]
test = sales[sales['month'] >= '2026-01-01']
print({'train': len(train), 'validation': len(validation), 'test': len(test)})

{'train': 864, 'validation': 216, 'test': 108}


## 3. Estacionalitat i diferències comercials

Compareu mesos, anys i productes. Una correlació en dades simulades no demostra una causa al món real.

In [5]:
train['month_number'] = train['month'].dt.month
seasonality = train.pivot_table(index='month_number', columns='product_line', values='sales_units', aggfunc='mean').round(1)
seasonality

product_line,electric,mountain,urban
month_number,,,
1,43.2,62.9,114.5
2,48.4,75.4,127.8
3,55.1,92.2,141.5
4,66.4,119.5,162.5
5,66.8,110.5,144.8
6,67.5,113.3,150.8
7,76.2,131.4,177.5
8,61.2,101.4,142.1
9,57.2,86.2,133.9


In [6]:
train.groupby(['product_line', 'channel', 'region'])['sales_units'].agg(['mean','std','min','max']).round(2)

mean    std  min  max
product_line channel   region                         
electric     ecommerce coast    66.98  19.45   33  118
                       inland   47.94  11.92   23   84
                       metro    75.21  18.18   36  130
             store     coast    55.71  15.61   34   97
                       inland   37.00   8.94   17   54
                       metro    65.52  14.35   39  103
mountain     ecommerce coast   105.56  28.32   59  179
                       inland   73.38  20.68   38  128
                       metro   117.98  32.51   52  190
             store     coast    90.21  23.62   53  156
                       inland   63.12  17.40   38  104
                       metro   105.19  32.73   48  181
urban        ecommerce coast   163.67  31.24  110  238
                       inland  111.73  23.82   75  178
                       metro   182.54  29.83  129  271
             store     coast   135.94  29.74   82  217
                       inland   93.35  14.97   58  125
                       metro   154.65  31.33  100  234

In [7]:
train.assign(year=train['month'].dt.year).groupby(['year','product_line'])['sales_units'].sum().unstack()

product_line,electric,mountain,urban
year,,,
2021,3577,6483,9870
2022,4191,6512,10050
2023,4374,6819,10396
2024,4579,6847,10094


## 4. Estoc i informació disponible

`stockout`, estoc final, sessions i ingressos només es coneixen al tancament. Són útils per analitzar, però no com a predictors del mateix mes.

In [8]:
print('Vendes superiors a estoc:', (sales['sales_units'] > sales['stock_units']).sum())
print('Estoc final incoherent:', (sales['closing_stock_units'] != sales['stock_units'] - sales['sales_units']).sum())
train.groupby('product_line')['stockout'].mean().rename('proporcio_ruptures')

Vendes superiors a estoc: 0
Estoc final incoherent: 0


product_line
electric    0.194444
mountain    0.194444
urban       0.218750
Name: proporcio_ruptures, dtype: float64

## 5. Clients: un problema separat

No uniu clients amb vendes per ciutat, canal o ordre de files. Són mostres independents. La fotografia de clients és del 30/06/2026.

In [9]:
customers[['annual_spend','avg_order_value','orders_last_12m','returns_ratio','days_since_last_order']].describe().round(2)

,annual_spend,avg_order_value,orders_last_12m,returns_ratio,days_since_last_order
count,1970.00,2000.00,2000.00,2000.00,1980.00
mean,1376.51,254.91,5.25,0.07,71.83
std,1618.56,266.43,2.99,0.13,80.34
min,35.31,19.06,1.00,0.00,0.00
25%,321.96,93.96,3.00,0.00,17.00
50%,758.02,145.56,5.00,0.00,44.00
75%,1864.55,289.39,7.00,0.10,95.00
max,13968.60,1898.82,17.00,1.00,364.00


## Entrega

Escriviu una pregunta de predicció, dues hipòtesis sobre entrenament, el tractament proposat dels nuls i una llista de predictors disponibles abans del mes. Expliqueu per què no es pot usar `revenue` per predir `sales_units` del mateix mes.

Pregunta 1: ¿Podemos predecir cuántas unidades de cada línea de producto se venderán en cada región y canal durante el próximo mes, utilizando únicamente la información histórica de ventas y stock disponible hasta el último día del mes anterior?

Hipótesis 1 (Estacionalidad y Canal): Las ventas de bicicletas y accesorios presentan una fuerte estacionalidad (picos en primavera/verano), pero esta variación estacional es mucho más pronunciada en el canal físico (retail) que en el canal online (ecommerce), donde las ventas son más constantes a lo largo del año.

Hipótesis 2 (Impacto de las roturas de stock): Las roturas de stock (stockout) en un mes determinado reducen significativamente el volumen de ventas esperado para ese producto en ese mes, y generan un "efecto rebote" (aumento de ventas) en el mes inmediatamente posterior cuando se repone el inventario.

No puedes usar los ingresos (revenue) de noviembre para predecir las unidades vendidas en noviembre porque los ingresos se calculan a partir de las unidades vendidas. Cuando empiece noviembre, tú no sabrás cuáles son los ingresos totales de ese mes. Si usas un dato que solo se conoce al finalizar el mes para predecir el comportamiento de ese mismo mes, tu modelo hará trampas y será inútil en la vida real. Solo puedes usar variables que conozcas antes de que empiece el mes a predecir